In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
data = pd.read_csv('insurance.csv')

In [ ]:
data

In [ ]:
data.shape


In [ ]:
data.head()

In [ ]:
data.tail()

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
data.isnull().sum()

In [ ]:
data.columns

In [ ]:
numeric_columns = ['age', 'bmi', 'children', 'charges']
for columns in numeric_columns:
    plt.figure(figsize=(6,4))
    sns.histplot(data[columns] , kde = True , bins = 20)

In [ ]:
sns.countplot(x = data['children'])

In [ ]:
sns.countplot(x = data['sex'])

In [ ]:
sns.countplot(x = data['smoker'])

In [ ]:
for col in numeric_columns:
    plt.figure(figsize=(6,4))
    sns.boxplot(x = data[col])

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(data.corr(numeric_only= True), annot=True)
plt.title('Correlation Heatmap')
plt.show()


In [ ]:
#Data cleaning and preprocessing

In [ ]:
df_cleaned = data.copy()
df_cleaned.head()

In [ ]:
df_cleaned.drop_duplicates(inplace= True)
df_cleaned.shape

In [ ]:
df_cleaned.isnull().sum()

In [ ]:
df_cleaned.dtypes

In [ ]:
df_cleaned['sex'].value_counts()

In [ ]:
df_cleaned['sex'] = df_cleaned['sex'].map({'female':1, 'male':0})
df_cleaned['sex'].value_counts()

In [ ]:
df_cleaned['smoker'] = df_cleaned['smoker'].map({'yes':1 , 'no':0})
df_cleaned['smoker'].value_counts()

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned.rename(columns = {'smoker':'is_smoker' , 'sex' : 'is_female'}, inplace= True)
df_cleaned.head()

In [ ]:
df_cleaned['region'].value_counts()

In [ ]:
df_cleaned = pd.get_dummies(df_cleaned , columns = ['region'])
df_cleaned.head()

In [ ]:
df_cleaned = df_cleaned.astype(int)

In [ ]:
df_cleaned

In [ ]:
#Feature Engineering and Extraction

In [ ]:
sns.histplot(df_cleaned['bmi'])

In [ ]:
df_cleaned['bmi_category'] = pd.cut(
    df_cleaned['bmi'],
    bins = [0, 18.5, 24.9, 29.9, float('inf')],
    labels = ['Underweight', 'Normal', 'Overweight', 'Obese']
)


In [ ]:
df_cleaned

In [ ]:
# Re-initializing df_cleaned and reapplying all transformations to ensure data integrity.
# This addresses the KeyError and the loss of other columns caused by prior steps.

df_cleaned = data.copy()
df_cleaned.drop_duplicates(inplace=True)

df_cleaned['is_female'] = df_cleaned['sex'].map({'female':1, 'male':0})
df_cleaned['is_smoker'] = df_cleaned['smoker'].map({'yes':1 , 'no':0})

df_cleaned.drop(columns=['sex', 'smoker'], inplace=True)

df_cleaned = pd.get_dummies(df_cleaned , columns = ['region'], drop_first=False) # Using drop_first to avoid multicollinearity

df_cleaned['bmi_category'] = pd.cut(
    df_cleaned['bmi'],
    bins = [0, 18.5, 24.9, 29.9, float('inf')],
    labels = ['Underweight', 'Normal', 'Overweight', 'Obese']
)

# Now, one-hot encode bmi_category and concatenate with existing df_cleaned
df_cleaned = pd.get_dummies(df_cleaned, columns=['bmi_category'])

# Convert boolean columns (from get_dummies) to int (0 or 1) for consistency
for col in df_cleaned.columns:
    if df_cleaned[col].dtype == 'bool':
        df_cleaned[col] = df_cleaned[col].astype(int)

# Display the cleaned DataFrame to verify
df_cleaned.info()
df_cleaned.head()

In [ ]:
df_cleaned.columns

In [ ]:
from sklearn.preprocessing import StandardScaler
cols = ["age" , "bmi" , "children"]
scaler = StandardScaler()
df_cleaned[cols] = scaler.fit_transform(df_cleaned[cols])


In [ ]:
from scipy.stats import pearsonr
selected_features =['age', 'bmi', 'children', 'is_female', 'is_smoker',
       'region_northeast', 'region_northwest', 'region_southeast',
       'region_southwest', 'bmi_category_Underweight', 'bmi_category_Normal',
       'bmi_category_Overweight', 'bmi_category_Obese']

In [ ]:
correlations = {
    feature: pearsonr(df_cleaned[feature], df_cleaned['charges'])[0]
    for feature in selected_features
}
correlation_df = pd.DataFrame(list(correlations.items()), columns=['Feature', 'Correlation'])
correlation_df = correlation_df.sort_values(by='Correlation', ascending=False)
correlation_df

In [ ]:
cat_features = ['region_northeast', 'region_northwest', 'region_southeast',
       'region_southwest', 'bmi_category_Underweight', 'bmi_category_Normal',
       'bmi_category_Overweight', 'bmi_category_Obese' , 'is_female' , 'is_smoker']

In [ ]:
from scipy.stats import chi2_contingency
import pandas as pd
alpha = 0.05
df_cleaned['charges_bin'] = pd.qcut(df_cleaned['charges'], q=4, labels=False)
chi2_results = {}


In [ ]:
for col in cat_features:
    contingency = pd.crosstab(df_cleaned[col], df_cleaned['charges_bin'])
    chi2_stat, p_val, _, _ = chi2_contingency(contingency)
    decision = 'Reject Null (Keep feature)' if p_val < alpha else 'Accept Null (Drop feature)'
    chi2_results[col] = {
        'chi2 Statistic': chi2_stat,
        'p-value': p_val,
        'Decision': decision
    }

chi2_diff = pd.DataFrame(chi2_results).T
chi2_diff = chi2_diff.sort_values(by='p-value')
chi2_diff

In [ ]:
final_df = df_cleaned[['age' , 'is_female' , 'bmi' , 'children' , 'is_smoker' , 'charges' , 'region_southeast' , 'bmi_category_Obese']]

In [ ]:
final_df

In [ ]:
from sklearn.model_selection import train_test_split
X = final_df.drop('charges' , axis = 1)
Y = final_df['charges']
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
X_train

In [ ]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train , Y_train)

In [ ]:
Y_pred = model.predict(X_test)
np.set_printoptions(suppress = True , precision=2)

In [ ]:
from sklearn.metrics import r2_score
r2 = r2_score(Y_test , Y_pred)
r2


In [ ]:
n = X_test.shape[0]
p = X_test.shape[1]
adjusted_r2 = 1 - ((1 - r2) * (n - 1) / (n - p - 1))
adjusted_r2